In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import random

PATH = '/data/'
seed = 2025
dataset_name = 'ml-1m'
saved_file_path = os.path.join(PATH, f'{dataset_name}/saved_data')

ratings_df = pd.read_csv(os.path.join(PATH, f'{dataset_name}/ratings.dat'), delimiter='::',encoding='ISO-8859-1', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_df.head()

/tmp/ipykernel_3039069/4266010308.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings_df = pd.read_csv(os.path.join(PATH, f'{dataset_name}/ratings.dat'), delimiter='::',encoding='ISO-8859-1', names=['user_id', 'item_id', 'rating', 'timestamp'])


,user_id,item_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## Get user history list with negative sample

In [ ]:
ratings_df['datetime'] = pd.to_datetime(ratings_df['timestamp'], unit='s').dt.strftime('%y%m%d')
sorted_df = ratings_df.sort_values(by=['user_id', 'datetime'])
sorted_df.head()

,user_id,item_id,rating,timestamp,datetime
0,1,1193,5,978300760,001231
1,1,661,3,978302109,001231
2,1,914,3,978301968,001231
3,1,3408,4,978300275,001231
5,1,1197,3,978302268,001231


In [ ]:
grouped = sorted_df.groupby(['user_id', 'datetime'])
list_of_lists = [group['item_id'].tolist() for _, group in grouped]
filtered_lists = [inner_list for inner_list in list_of_lists if 5 < len(inner_list)]
len(filtered_lists)

11929

In [9]:
import random
all_movie_ids_set = set([movie_id for inner_list in filtered_lists for movie_id in inner_list])

movies_except_last = []
last_movie_ids = []
shuffled_movie_ids_with_last = []

max_history_length = 25
for inner_list in filtered_lists:
    except_last = inner_list[-max_history_length:-1]
    movies_except_last.append(except_last)
    
    last_movie_id = inner_list[-1]
    last_movie_ids.append(last_movie_id)
    
    possible_ids = list(all_movie_ids_set.difference(except_last))
    random_ids = random.sample(possible_ids, 49)
    
    random_ids.append(last_movie_id)
    random.shuffle(random_ids)
    shuffled_movie_ids_with_last.append(random_ids)

In [ ]:
from sklearn.model_selection import train_test_split

test_size = 0.2
zipped_lists = list(zip(movies_except_last, last_movie_ids, shuffled_movie_ids_with_last))
train_zipped, test_zipped = train_test_split(zipped_lists, test_size=test_size, random_state=seed)


import os

if not os.path.exists(saved_file_path):
    os.makedirs(saved_file_path)

with open(os.path.join(saved_file_path, f'{dataset_name}_train.txt')), 'wb') as file:  # Further split train into train and val when training!
    pickle.dump(train_zipped, file)
with open(os.path.join(saved_file_path, f'{dataset_name}_test.txt')), 'wb') as file:
    pickle.dump(test_zipped, file)

In [4]:
movie_df = pd.read_csv(os.path.join(PATH, f'{dataset_name}/movies.dat'), delimiter='::',encoding='ISO-8859-1', names=['item_id', 'title', 'genres'])
movie_df

/tmp/ipykernel_2371298/3003223545.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  movie_df = pd.read_csv(os.path.join(PATH, f'{dataset_name}/movies.dat'), delimiter='::',encoding='ISO-8859-1', names=['item_id', 'title', 'genres'])


,item_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama
